# SLM-SAM2 — Sheffield Dataset Evaluation

Computes per-muscle Dice / Hausdorff metrics for SLM-SAM2 segmentations on the **Sheffield** dataset.

- **Predictions**: `../sheffield_segs/Aug_N_slmsam2.npz` (keys: MuscleNm_L / MuscleNm_R)
- **Ground truth**: `../../sheffeld/20440203/Aug_N_segmentations.dcm` (labels 1–37, bilateral)

Sheffield GT labels are **bilateral** — _L and _R keys are OR-combined before comparison.
Prompts were seeded from MuscleMap WB Sheffield segmentations.

In [ ]:
import glob, os, re
import numpy as np
import pandas as pd
import SimpleITK as sitk
import pydicom
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
BOUNDARY_DISTANCE = 1
GT_DIR     = os.path.join('..', '..', 'sheffeld', '20440203')
SEG_DIR    = os.path.join('..', 'sheffield_segs')
RESULT_DIR = 'results_sheffield'
ALGO_TAG   = 'slmsam'
os.makedirs(RESULT_DIR, exist_ok=True)

# (muscle_name, sheffield_gt_label, npz_keys)
# SLM-SAM2 NPZ keys use _L / _R suffix (from MuscleMap WB 7xxx label names).
MUSCLES = [
    ('adductor_magnus',    3,  ['Adductor_Magnus_L',    'Adductor_Magnus_R'   ]),
    ('biceps_femoris_long', 5, ['Biceps_Femoris_L',     'Biceps_Femoris_R'    ]),
    ('gracilis',           16, ['Gracilis_L',           'Gracilis_R'          ]),
    ('rectus_femoris',     27, ['Rectus_Femoris_L',     'Rectus_Femoris_R'    ]),
    ('sartorius',          28, ['Sartorius_L',          'Sartorius_R'         ]),
    ('semimembranosus',    29, ['Semimembranosus_L',    'Semimembranosus_R'   ]),
    ('semitendinosus',     30, ['Semitendinosus_L',     'Semitendinosus_R'    ]),
    ('vastus_intermedius', 35, ['Vastus_Intermedius_L', 'Vastus_Intermedius_R']),
    ('vastus_lateralis',   36, ['Vastus_Lateralis_L',   'Vastus_Lateralis_R'  ]),
    ('vastus_medialis',    37, ['Vastus_Medialis_L',    'Vastus_Medialis_R'   ]),
]

def read_gt(idx):
    ds  = pydicom.dcmread(os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm'))
    raw = ds.pixel_array.astype(np.float32)
    if raw.ndim == 2: raw = raw[np.newaxis]
    labeled = np.round(raw * 37.0 / 255.0).astype(np.int32)
    labeled[raw == 0] = 0
    return np.clip(labeled, 0, 37)

def get_spacing(idx):
    ds = pydicom.dcmread(os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm'))
    ps = getattr(ds, 'PixelSpacing', [1.0, 1.0])
    st = float(getattr(ds, 'SliceThickness', 1.0))
    return (float(ps[1]), float(ps[0]), st)

seg_files = sorted(glob.glob(os.path.join(SEG_DIR, 'Aug_*_slmsam2.npz')),
                   key=lambda p: int(re.search(r'Aug_(\d+)', p).group(1)))
print(f'Found {len(seg_files)} NPZ files')
if seg_files:
    s = np.load(seg_files[0])
    print(f'Sample keys: {sorted(s.files)}')

In [ ]:
def evaluate_muscle(muscle_name, sheffield_label, npz_keys):
    results = []
    for seg_path in seg_files:
        idx = re.search(r'Aug_(\d+)', seg_path.replace('\\', '/')).group(1)
        gt_path = os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm')
        if not os.path.exists(gt_path):
            print(f'  [skip] GT missing: Aug_{idx}'); continue
        gt_arr  = read_gt(idx)
        spacing = get_spacing(idx)
        gt_bin  = (gt_arr == sheffield_label).astype(np.uint8)

        npz_data = np.load(seg_path)
        pred_arr = np.zeros(gt_arr.shape, dtype=np.uint8)
        for key in npz_keys:
            if key in npz_data:
                pred_arr |= npz_data[key].astype(np.uint8)
            else:
                print(f'  [Aug_{idx}] key "{key}" not in NPZ')

        gt_s = sitk.GetImageFromArray(gt_bin);   gt_s.SetSpacing(spacing)
        pr_s = sitk.GetImageFromArray(pred_arr); pr_s.SetSpacing(spacing)
        dice_f = sitk.LabelOverlapMeasuresImageFilter(); dice_f.Execute(gt_s, pr_s)
        if gt_bin.sum() > 0 and pred_arr.sum() > 0:
            hd_f = sitk.HausdorffDistanceImageFilter(); hd_f.Execute(gt_s, pr_s)
            hd = hd_f.GetHausdorffDistance()
        else:
            hd = np.nan
        results.append({
            'sample': f'Aug_{idx}',
            f'{muscle_name}_dice':                  dice_f.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_f.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_f.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_f.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_f.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_bin.astype(float), pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_bin.astype(float), pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_bin.astype(float)),
        })
    df = pd.DataFrame(results)
    csv_path = os.path.join(RESULT_DIR, f'df_{muscle_name}_{ALGO_TAG}_sheffield.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df

dfs = {}
for muscle_name, sheffield_label, npz_keys in MUSCLES:
    print(f'\n── {muscle_name}  (GT={sheffield_label}, keys={npz_keys}) ──')
    dfs[muscle_name] = evaluate_muscle(muscle_name, sheffield_label, npz_keys)
print('\nDone.')

In [ ]:
from IPython.display import display
summary_rows = []
for muscle_name, df in dfs.items():
    if df.empty: continue
    summary_rows.append({
        'muscle': muscle_name, 'n': len(df),
        'dice_mean': df[f'{muscle_name}_dice'].mean(), 'dice_std': df[f'{muscle_name}_dice'].std(),
        'hausdorff_mean': df[f'{muscle_name}_hausdorff'].mean(), 'hausdorff_std': df[f'{muscle_name}_hausdorff'].std(),
    })
summary = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, f'summary_{ALGO_TAG}_sheffield.csv')
summary.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
display(summary.round(4))